In [1]:
import pandas as pd
import numpy as np

In [3]:
import json
from pathlib import Path
 
import pandas as pd
 
TABLES = ["employees", "projects", "project_details", "hours_monthly", "hours_total"]
 
 
def iter_companies(root="companies"):
    for folder in sorted(Path(root).glob("company_*")):
        t = {name: pd.read_csv(folder / f"{name}.csv") for name in TABLES}
        meta = json.loads((folder / "company.json").read_text(encoding="utf-8"))
        yield folder.name, t, meta

In [6]:
folder, tables, metadata = next(iter_companies("companies"))
tables["employees"].head()

,employee,employee_name,role,department,level,salary,start_date,end_date,max_concurrent,project_preference
0,E0,Sara Petrov,Remuneration Analyst,HR,2,49500,2024-11-19,NaN,0,none
1,E1,Noah Brooks,Analyst,Delivery,2,46000,2024-11-10,NaN,3,any
2,E2,Ruth Dubois,Team Lead,Delivery,4,83500,2022-08-09,NaN,3,any
3,E3,Jun Ahmed,Head of Department,Delivery,5,97500,2018-07-24,NaN,4,long


In [7]:
# table:
# employee ID, salary, start_date, end_date,

tables["hours_monthly"].head()

,employee,project,month,hours
0,E2,P1,2025-02,43
1,E2,P1,2025-03,43
2,E2,P1,2025-04,43
3,E2,P1,2025-05,42
4,E2,P1,2025-06,39


In [8]:
tables["projects"].head()

,project,revenue,start_month,end_month
0,P1,397000,2025-02,2025-08


In [9]:
tables["project_details"].head()

,project,project_name,client,client_type
0,P1,Battery storage investment case,Dunmere Water Services,Water utility


In [10]:
tables["hours_total"].head()

,employee,P1
0,E0,0
1,E1,772
2,E2,282
3,E3,0


In [44]:
def get_project_contribution_bonuses(company_tables):
    project_hours = tables["hours_monthly"].groupby(["employee","project"])["hours"].sum().reset_index()
    total_hours = project_hours.groupby(["project"])["hours"].sum().reset_index()
    project_hours = pd.merge(project_hours, tables["projects"], how='left', on="project")
    project_hours = pd.merge(project_hours, total_hours, how='left', on='project')
    project_hours["bonus"] = 100000
    project_hours = project_hours.rename(columns={"hours_x": "employee_hours", "hours_y": "total_project_hours"})
    employee_salary = tables["employees"][["employee", "salary"]]
    project_hours = pd.merge(project_hours, employee_salary, how='left', on='employee')
    project_hours["percent_hours"] = project_hours["employee_hours"] / project_hours["total_project_hours"]
    project_hours["revenue_share"] = project_hours["percent_hours"] * project_hours["revenue"]
    employee_bonuses = project_hours.groupby("employee")["revenue_share"].sum().reset_index()
    total_revenue_share = employee_bonuses["revenue_share"].sum()
    employee_bonuses["bonus_share"] = employee_bonuses["revenue_share"] / total_revenue_share
    employee_bonuses["project_contribution_bonus"] = 100000 * employee_bonuses["bonus_share"]
    employee_bonuses = employee_bonuses[["employee", "project_contribution_bonus"]]
    
    return employee_bonuses

In [27]:
project_hours = tables["hours_monthly"].groupby(["employee","project"])["hours"].sum().reset_index()
total_hours = project_hours.groupby(["project"])["hours"].sum().reset_index()
project_hours = pd.merge(project_hours, tables["projects"], how='left', on="project")
project_hours = pd.merge(project_hours, total_hours, how='left', on='project')
project_hours["bonus"] = 100000
project_hours = project_hours.rename(columns={"hours_x": "employee_hours", "hours_y": "total_project_hours"})


project_hours.head()

,employee,project,employee_hours,revenue,start_month,end_month,total_hours,bonus
0,E1,P1,772,397000,2025-02,2025-08,1054,100000
1,E2,P1,282,397000,2025-02,2025-08,1054,100000


In [30]:
employee_salary = tables["employees"][["employee", "salary"]]
project_hours = pd.merge(project_hours, employee_salary, how='left', on='employee')
project_hours.head()

,employee,project,employee_hours,revenue,start_month,end_month,total_hours,bonus,salary
0,E1,P1,772,397000,2025-02,2025-08,1054,100000,46000
1,E2,P1,282,397000,2025-02,2025-08,1054,100000,83500


In [32]:
project_hours["percent_hours"] = project_hours["employee_hours"] / project_hours["total_hours"]
project_hours["revenue_share"] = project_hours["percent_hours"] * project_hours["revenue"]
project_hours.head()

,employee,project,employee_hours,revenue,start_month,end_month,total_hours,bonus,salary,percent_hours,revenue_share
0,E1,P1,772,397000,2025-02,2025-08,1054,100000,46000,0.732448,290781.783681
1,E2,P1,282,397000,2025-02,2025-08,1054,100000,83500,0.267552,106218.216319


In [36]:
employee_bonuses = project_hours.groupby("employee")["revenue_share"].sum().reset_index()
employee_bonuses.head()

,employee,revenue_share
0,E1,290781.783681
1,E2,106218.216319


In [38]:
# project contribution bonus is calculated purely from the time an employee spends on project work
total_revenue_share = employee_bonuses["revenue_share"].sum()
employee_bonuses["bonus_share"] = employee_bonuses["revenue_share"] / total_revenue_share
employee_bonuses["project_contribution_bonus"] = 100000 * employee_bonuses["bonus_share"]
employee_bonuses.head()

,employee,revenue_share,bonus_share,employee_bonus,project_contribution_bonus
0,E1,290781.783681,0.732448,73244.781784,73244.781784
1,E2,106218.216319,0.267552,26755.218216,26755.218216


In [39]:
emp = tables["employees"]
current_employees = emp[emp["end_date"].isna()]
current_employees.head()

,employee,employee_name,role,department,level,salary,start_date,end_date,max_concurrent,project_preference
0,E0,Sara Petrov,Remuneration Analyst,HR,2,49500,2024-11-19,NaN,0,none
1,E1,Noah Brooks,Analyst,Delivery,2,46000,2024-11-10,NaN,3,any
2,E2,Ruth Dubois,Team Lead,Delivery,4,83500,2022-08-09,NaN,3,any
3,E3,Jun Ahmed,Head of Department,Delivery,5,97500,2018-07-24,NaN,4,long


In [45]:
test_bonus = get_project_contribution_bonuses(tables)
test_bonus.head()

,employee,project_contribution_bonus
0,E1,73244.781784
1,E2,26755.218216


In [46]:
def get_project_contribution_bonuses(tables, pool=100000, current_only=False):
    hours = tables["hours_monthly"]
    if current_only:
        current = tables["employees"].loc[tables["employees"]["end_date"].isna(), "employee"]
        hours = hours[hours["employee"].isin(current)]

    project_hours = (hours.groupby(["employee", "project"], as_index=False)["hours"].sum()
                     .rename(columns={"hours": "employee_hours"})
                     .merge(tables["projects"][["project", "revenue"]], on="project", how="left"))
    project_hours["total_project_hours"] = project_hours.groupby("project")["employee_hours"].transform("sum")
    project_hours["revenue_share"] = (project_hours["employee_hours"] / project_hours["total_project_hours"]
                                      * project_hours["revenue"])

    bonuses = project_hours.groupby("employee", as_index=False)["revenue_share"].sum()
    bonuses["project_contribution_bonus"] = pool * bonuses["revenue_share"] / bonuses["revenue_share"].sum()
    return bonuses[["employee", "project_contribution_bonus"]]


def get_project_contribution_bonuses_current(tables, pool=100000):
    return get_project_contribution_bonuses(tables, pool, current_only=True)

In [47]:
test_bonus = get_project_contribution_bonuses(tables)
test_bonus.head()

,employee,project_contribution_bonus
0,E1,73244.781784
1,E2,26755.218216


In [48]:
test_bonus_current = get_project_contribution_bonuses_current(tables)
test_bonus_current.head()

,employee,project_contribution_bonus
0,E1,73244.781784
1,E2,26755.218216


In [53]:
import pandas as pd

ANALYST = "E0"   # the Remuneration Analyst


# ---------------------------------------------------------------- helpers
def _people_and_hours(company_tables, current_only):
    """Employees and monthly hours; with current_only, anyone with an end_date is dropped."""
    emp, hours = company_tables["employees"], company_tables["hours_monthly"]
    if current_only:
        emp = emp[emp["end_date"].isna()]
        hours = hours[hours["employee"].isin(emp["employee"])]
    return emp, hours


def _revenue_shares(hours, projects, weights=None):
    """Each person's total share of project revenue (a Series indexed by employee).

    A person's share of a project is their hours / everyone's hours on it, times its revenue.
    weights (a Series indexed by employee, e.g. salary) scales each person's hours first.
    """
    ph = (hours.groupby(["employee", "project"], as_index=False)["hours"].sum()
          .merge(projects[["project", "revenue"]], on="project", how="left"))
    ph["weighted"] = ph["hours"] if weights is None else ph["hours"] * ph["employee"].map(weights)
    ph["share"] = ph["weighted"] / ph.groupby("project")["weighted"].transform("sum") * ph["revenue"]
    return ph.groupby("employee")["share"].sum()


def _to_bonuses(shares, pool, column):
    """Split pool in proportion to shares (a Series indexed by employee)."""
    return (pool * shares / shares.sum()).rename(column).reset_index()


# ------------------------------------------- project contribution (your logic)
def get_project_contribution_bonuses(company_tables, pool=100000, current_only=False):
    """current_only=True drops leavers first, so each project's revenue is shared among
    the people still at the company who worked on it."""
    _, hours = _people_and_hours(company_tables, current_only)
    shares = _revenue_shares(hours, company_tables["projects"])
    return _to_bonuses(shares, pool, "project_contribution_bonus")


def get_project_contribution_bonuses_current(company_tables, pool=100000):
    return get_project_contribution_bonuses(company_tables, pool, current_only=True)


def get_project_contribution_bonuses_current_rescaled(company_tables, pool=100000):
    """Leavers' hours still count towards each project's total, so their revenue share is
    not passed to colleagues. Leavers are then dropped and the rest rescaled to the pool."""
    shares = _revenue_shares(company_tables["hours_monthly"], company_tables["projects"])
    current, _ = _people_and_hours(company_tables, current_only=True)
    return _to_bonuses(shares[shares.index.isin(current["employee"])], pool, "project_contribution_bonus")


# ------------------------------------------------ (a) salary-weighted projects
def get_salary_weighted_project_bonuses(company_tables, pool=100000, current_only=False):
    """Same as project contribution, but an hour counts in proportion to its worker's salary."""
    _, hours = _people_and_hours(company_tables, current_only)
    salary = company_tables["employees"].set_index("employee")["salary"]
    shares = _revenue_shares(hours, company_tables["projects"], weights=salary)
    return _to_bonuses(shares, pool, "salary_weighted_project_bonus")


# ------------------------------------------------------ (b), (c) pure salary
def get_salary_bonuses(company_tables, pool=100000, current_only=False, include_analyst=True):
    """Pool split in proportion to salary across everyone in the employees table
    (project staff, support staff and, if include_analyst, the Remuneration Analyst)."""
    emp, _ = _people_and_hours(company_tables, current_only)
    if not include_analyst:
        emp = emp[emp["employee"] != ANALYST]
    return _to_bonuses(emp.set_index("employee")["salary"], pool, "salary_bonus")


def get_salary_bonuses_excluding_analyst(company_tables, pool=100000, current_only=False):
    return get_salary_bonuses(company_tables, pool, current_only, include_analyst=False)


# ------------------------------------------------- (d) support staff set share
def get_support_share_bonuses(company_tables, pool=100000, support_share=0.2, current_only=False,
                              support_split="salary", include_analyst=True):
    """Support staff share support_share of the pool, split by salary (or support_split="equal").
    The rest goes to everyone with project hours, by project contribution.

    Support staff = anyone with no project hours at all: the non-project departments, the
    Remuneration Analyst, and project staff who logged no hours. A company in which everyone
    logged hours pays the whole pool by project contribution.
    include_analyst=False leaves the Remuneration Analyst out altogether (they get nothing).
    """
    emp, hours = _people_and_hours(company_tables, current_only)
    if not include_analyst:
        emp = emp[emp["employee"] != ANALYST]
    support = emp[~emp["employee"].isin(hours["employee"])].set_index("employee")["salary"]
    if support.empty:
        support_share = 0
    project_part = _to_bonuses(_revenue_shares(hours, company_tables["projects"]),
                               pool * (1 - support_share), "support_share_bonus")
    if support.empty:
        return project_part
    if support_split not in ("salary", "equal"):
        raise ValueError('support_split must be "salary" or "equal"')
    weights = support if support_split == "salary" else support * 0 + 1
    support_part = _to_bonuses(weights, pool * support_share, "support_share_bonus")
    return pd.concat([project_part, support_part], ignore_index=True)


# ------------------------------------------ (e) as (d), the analyst gets nothing
def get_support_share_bonuses_excluding_analyst(company_tables, pool=100000, support_share=0.2,
                                                current_only=False, support_split="salary"):
    """Same as (d), but E0 is left out: the support share is split among the other support staff."""
    return get_support_share_bonuses(company_tables, pool, support_share, current_only,
                                     support_split, include_analyst=False)


# --------------------- (f), (g) salary target adjusted by a bounded contribution rating
def get_performance_adjusted_salary_bonuses(company_tables, pool=100000, current_only=False,
                                            include_analyst=True, floor=0.5, cap=1.5):
    """Each person's weight is their salary times a contribution multiplier; the pool is then
    split in proportion to those weights.

    The multiplier is the person's project revenue share divided by the average revenue share
    of people with project hours, limited to the range floor..cap. People with no project hours
    get 1.0 (they are treated as meeting expectations).
    """
    emp, hours = _people_and_hours(company_tables, current_only)
    if not include_analyst:
        emp = emp[emp["employee"] != ANALYST]
    shares = _revenue_shares(hours, company_tables["projects"])        # only people with hours
    multiplier = (shares / shares.mean()).clip(floor, cap) if len(shares) else shares
    salary = emp.set_index("employee")["salary"]
    weights = salary * multiplier.reindex(salary.index).fillna(1.0)
    return _to_bonuses(weights, pool, "performance_adjusted_salary_bonus")


def get_performance_adjusted_salary_bonuses_excluding_analyst(company_tables, pool=100000,
                                                              current_only=False, floor=0.5, cap=1.5):
    return get_performance_adjusted_salary_bonuses(company_tables, pool, current_only,
                                                   include_analyst=False, floor=floor, cap=cap)


# ------------------------------------------------------- everything together
def get_all_bonuses(company_tables, pool=100000, current_only=False):
    """One row per employee in the employees table, one column per bonus definition.

    Someone who gets nothing under a definition (no project hours, a leaver under a
    current-only definition, the analyst under (c) and (d), ...) has 0 in that column.
    current_only=True applies the leaver filter to all the other columns too; the two
    *_current columns always filter leavers.
    """
    columns = [
        get_project_contribution_bonuses(company_tables, pool, current_only),
        get_project_contribution_bonuses_current(company_tables, pool)
            .rename(columns={"project_contribution_bonus": "project_contribution_bonus_current"}),
        get_project_contribution_bonuses_current_rescaled(company_tables, pool)
            .rename(columns={"project_contribution_bonus": "project_contribution_bonus_current_rescaled"}),
        get_salary_weighted_project_bonuses(company_tables, pool, current_only),
        get_salary_bonuses(company_tables, pool, current_only),
        get_salary_bonuses_excluding_analyst(company_tables, pool, current_only)
            .rename(columns={"salary_bonus": "salary_bonus_excluding_analyst"}),
        get_support_share_bonuses(company_tables, pool, current_only=current_only),
        get_support_share_bonuses_excluding_analyst(company_tables, pool, current_only=current_only)
            .rename(columns={"support_share_bonus": "support_share_bonus_excluding_analyst"}),
        get_performance_adjusted_salary_bonuses(company_tables, pool, current_only),
        get_performance_adjusted_salary_bonuses_excluding_analyst(company_tables, pool, current_only)
            .rename(columns={"performance_adjusted_salary_bonus": "performance_adjusted_salary_bonus_excluding_analyst"}),
    ]
    all_bonuses = company_tables["employees"][["employee"]].copy()
    for column in columns:
        all_bonuses = all_bonuses.merge(column, on="employee", how="left")
    return all_bonuses.fillna(0)


In [54]:
final_test = get_all_bonuses(tables)
final_test.head()

,employee,project_contribution_bonus,project_contribution_bonus_current,project_contribution_bonus_current_rescaled,salary_weighted_project_bonus,salary_bonus,salary_bonus_excluding_analyst,support_share_bonus,support_share_bonus_excluding_analyst,performance_adjusted_salary_bonus,performance_adjusted_salary_bonus_excluding_analyst
0,E0,0.000000,0.000000,0.000000,0.000000,17902.350814,0.000000,6734.693878,0.000000,19107.069612,0.000000
1,E1,73244.781784,73244.781784,73244.781784,60129.700808,16636.528029,20264.317181,58595.825427,58595.825427,26010.781671,32154.579574
2,E2,26755.218216,26755.218216,26755.218216,39870.299192,30198.915009,36784.140969,21404.174573,21404.174573,17247.011602,21320.789739
3,E3,0.000000,0.000000,0.000000,0.000000,35262.206148,42951.541850,13265.306122,20000.000000,37635.137115,46524.630687


In [55]:
for folder, tables, metadata in iter_companies("companies"):
    bonuses = get_all_bonuses(tables)

In [56]:
bonuses.head()

,employee,project_contribution_bonus,project_contribution_bonus_current,project_contribution_bonus_current_rescaled,salary_weighted_project_bonus,salary_bonus,salary_bonus_excluding_analyst,support_share_bonus,support_share_bonus_excluding_analyst,performance_adjusted_salary_bonus,performance_adjusted_salary_bonus_excluding_analyst
0,E0,0.000000,0.000000,0.000000,0.000000,2147.170655,0.000000,1201.501877,0.000000,2299.113244,0.000000
1,E1,4152.566735,4322.654792,4428.250401,4731.920772,2572.131514,2628.571429,3322.053388,3322.053388,2744.826088,2809.417784
2,E2,5945.968971,6007.986381,6340.714349,8300.002702,3466.785954,3542.857143,4756.775177,4756.775177,5297.301704,5421.958674
3,E3,528.132127,584.039447,563.194152,728.815434,3176.023261,3245.714286,422.505701,422.505701,1700.385837,1740.399594
4,E4,1003.905698,1003.905698,1070.553731,598.740866,1476.179826,1508.571429,803.124558,803.124558,790.320178,808.918121
